# 🚀 MIVI-V2 Knowledge-Lean Sub-1B Fine-Tuning with Unsloth

> **Target Hardware:** Free Google Colab (T4 GPU, 16 GB VRAM)
> **Base Model:** `Qwen/Qwen2.5-0.5B-Instruct` (or `Qwen/Qwen3-0.6B`)
> **Training Method:** 4-bit QLoRA with Unsloth
> **Peak VRAM:** < 2.5 GB
> **Output:** GGUF `Q4_K_M` (~460 MB) for MIVI-V2 Pure Rust Server

In [ ]:
# 1. Install Unsloth and essential libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model in 4-bit with Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Supports up to 64k context with YaRN RoPE
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# 3. Add LoRA Adapters (Rank 16, Alpha 32)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Prepare Dataset
import json
from datasets import Dataset

# Upload your `datasets/mivi_sub1b_tuning_dataset.jsonl` or load directly
dataset_file = "mivi_sub1b_tuning_dataset.jsonl"

with open(dataset_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

formatted = []
for item in raw_data:
    text = tokenizer.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=False)
    formatted.append({"text": text})

train_dataset = Dataset.from_list(formatted)
print(f"Loaded {len(train_dataset)} training examples.")

In [ ]:
# 5. Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 250,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Export directly to GGUF (Q4_K_M)
model.save_pretrained_gguf("mivi-0.5b-tool-expert", tokenizer, quantization_method = "q4_k_m")
print("✅ Successfully exported GGUF model: mivi-0.5b-tool-expert-unsloth.Q4_K_M.gguf")